Here is the code where you put in H, M, G, model and measurement

In [1]:
#| default_exp prediction_module

In [1]:
#| export
import pickle
import pandas as pd 
from anthropmass.data_module import *
from anthropmass.anthro_module import *
from anthropmass.bambi_module import *

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install m2w64-toolchain`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


This function is normalizing the persons weight and height

In [2]:
#| export
def normalize_person(weight, height, gender):
    person=pd.DataFrame({'weightkg': [weight], 'stature': [height], 'Gender': [gender]})
    normalize(person, 'weightkg')
    normalize(person, 'stature')
    return person

In [3]:
#| export
def minus_mean_person(weight, height, gender):
    person=pd.DataFrame({'weightkg': [weight], 'stature': [height], 'Gender': [gender]})
    minus_mean(person, 'weightkg')
    minus_mean(person, 'stature')
    return person

This functions gets the pickled model

In [4]:
#| export
def get_pickled_model(kindofmodel:str, measurement:str):
    filepath = f'../output/anthro_models/{kindofmodel}/pickle_{measurement}_{kindofmodel}'
    with open(filepath,'rb') as file:
        model=pickle.load(file)
    return model

Predict_from_model uses the pickled model and the normalized person to predict the measurement. One of the arguments passed into this functions is "kind of model", and currently there is three different models to chose between. The three models are xgboost, bambi, and bambi with component as input. When using bambi with component as input, the component needs to passed as an argument into the functions, otherwise Army Reserve is set as default. The three different components to pass into the function are Regular Army, Army National Guard, and Army Reserve.

In [5]:
#| export
def predict_from_model(kindofmodel:str, measurement:str, w, h, g, c=False):
    
    pickledmodel = get_pickled_model(kindofmodel, measurement)
    #person = normalize_person(w, h, g)
    #person=pd.DataFrame({'weightkg': [w], 'stature': [h], 'Gender': [g]})
    person= minus_mean_person(w,h,g)

    if kindofmodel=='xgboost':
        return pickledmodel.predict(person)
    
    elif kindofmodel=='bambi':
        #model = model_bmb(measurement)
        formula = 'weightkg + stature + Gender'
        model = make_model_formula(measurement, formula)
        return predict_mean_bmb(pickledmodel, model, person, measurement)
    
    elif kindofmodel=='bambi_c':
        if c!= False:
            person['Component']=c
        else:
            person['Component']='Army Reserve'
        formula='0 + C(Gender) + Component + weightkg + stature'
        model = make_model_formula(measurement, formula)
        return predict_mean_bmb(pickledmodel, model, person, measurement)
    
    else:
        return 'wrong model name'

In [6]:
#| export
def add_to_df(df, measurement, pred):
    df[measurement]=pred
    return df

I dont know if we need to make it to csv?

In [7]:
#| export
def make_csv(data, name):
    data.to_csv(f'{name}.csv', index=False)

In the following function the function above is called upon and looped through for all passed measurements.

In [8]:
#| export
def make_predictions(kindofmodel:str, measurements:list, w, h, g, c=False):
    output=pd.DataFrame()
    for m in measurements:
        pred = predict_from_model(kindofmodel, m, w, h, g, c=False)
        add_to_df(output, m, pred)
    return output

Predict for group is a function that itterates through a dataframe with multiple people. THe argument n is used when you only want to predict a part of the dataframe. 

In [9]:
#| export
def predict_for_group(kindofmodel:str, measurements:list, group:dict, n=False):
    preds_all=pd.DataFrame()
    for index, row in group.iterrows():
        if index%10 == 0:
            print('index:', index)
        if kindofmodel == 'bambi_c':
            pred_row = make_predictions(kindofmodel, measurements, row['weightkg'], row['stature'], row['Gender'], row['Component'])
        else:
            pred_row = make_predictions(kindofmodel, measurements, row['weightkg'], row['stature'], row['Gender'])
        preds_all = pd.concat([preds_all, pred_row], ignore_index=True)
        if index == n:
            return preds_all
    return preds_all

In [10]:
col_all=measurement_names()
col=col_all[:2]
test= pd.read_csv('../data/processed/ANSURIItest.csv')
preds = predict_for_group('bambi', col_all, test, 1)

index: 0


KeyError: 'Gender'

In [14]:
test.columns[90:]


Index(['waistheightomphalion', 'weightkg', 'wristcircumference', 'wristheight',
       'Gender', 'Date', 'Installation', 'Component', 'Branch', 'PrimaryMOS',
       'SubjectsBirthLocation', 'SubjectNumericRace', 'Ethnicity', 'DODRace',
       'Age', 'Heightin', 'Weightlbs', 'WritingPreference'],
      dtype='object')

In [12]:
preds

,abdominalextensiondepthsitting,acromialheight,acromionradialelength,anklecircumference,axillaheight,balloffootcircumference,balloffootlength,biacromialbreadth,bicepscircumferenceflexed,bicristalbreadth,...,trochanterionheight,verticaltrunkcircumferenceusa,waistbacklength,waistbreadth,waistcircumference,waistdepth,waistfrontlengthsitting,waistheightomphalion,wristcircumference,wristheight
0,241.503349,1494.455572,347.187601,228.796659,1382.309832,254.026202,206.646389,422.506583,349.164348,275.139896,...,937.140246,1677.635305,481.0166,315.880974,902.741567,223.47138,400.779667,1105.268737,177.69712,876.886824


In [ ]:
make_csv(preds,'../output/anthro_results/predict_test_bambi')

In [56]:
import nbdev; nbdev.nbdev_export()